# DINOv2: Learning Robust Visual Features without Supervision

**Paper:** M. Oquab, T. Darcet, T. Moutakanni, H. V. Vo, M. Szafraniec, V. Khalidov, P. Fernandez, D. Haziza, F. Massa, A. El-Nouby, M. Assran, N. Ballas, W. Galvez, R. Beber, A. Joulin, H. Jégou, J. Mairal, P. Labatut, A. Joulin, and P. Bojanowski, "DINOv2: Learning Robust Visual Features without Supervision," *Transactions on Machine Learning Research (TMLR)*, 2024. [arXiv:2304.07193](https://arxiv.org/abs/2304.07193)

**Summary:** DINOv2 trains ViT backbones with self-supervised distillation at scale, introducing register tokens and LayerScale to obtain universal visual features that transfer without fine-tuning to tasks ranging from depth estimation to semantic segmentation.

## Architecture Overview

```
Input Image [B, 3, H, W]
        │
        ▼  PatchEmbed (Conv2d, patch_size=14)  ← DINOv2 uses p=14, not p=16
Patches [B, N, D]   N = (H/14)² = 256 for 224px
        │
        ▼  Build token sequence:
           [CLS] token prepended          → 1 token
           patch tokens                  → N tokens
           → cat → [B, N+1, D]
           + positional embedding (CLS + patches only)
        │
        ▼  Insert register tokens AFTER positional embedding:
           [CLS | reg₁ | reg₂ | reg₃ | reg₄ | P₁ | P₂ | ... | P_N]
           Total length: 1 + 4 + N = N+5
        │
        ▼  ┌─────────────────────────────────────────────────────────┐
           │  DINOBlock × depth                                       │
           │  ┌─────────────────────────────────────────────────────┐ │
           │  │  x = x + LayerScale_γ₁ · DINOAttn(LN(x))           │ │
           │  │  x = x + LayerScale_γ₂ · MLP(LN(x))                │ │
           │  │  DINOAttn: Flash Attention (F.scaled_dot_product)    │ │
           │  │  LayerScale: per-channel γ, init=1e-5               │ │
           │  └─────────────────────────────────────────────────────┘ │
           └─────────────────────────────────────────────────────────┘
        │
        ▼  Final LayerNorm
        │
        ▼  DISCARD register tokens (positions 1..4)
        │
        ├──► pooled_output = x[:, 0]                  (CLS)
        └──► patch_tokens  = x[:, 1+n_reg:]           (N patch tokens, reg discarded)
             feature_maps  = reshape → [B, D, H/14, W/14]
```

### DINO v1 vs DINOv2 — What Changed?
| Feature | DINO v1 | DINOv2 |
|---------|---------|--------|
| Register tokens | No | Yes (4 per image) |
| LayerScale | No | Yes (γ init=1e-5) |
| Flash Attention | No | Yes (PyTorch ≥2.0) |
| Patch size | 16 | 14 |
| Training loss | DINO (self-distillation) | DINO + iBOT + KoLeo |
| Data curation | ImageNet | Large curated 142M dataset |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig = plt.figure(figsize=(17, 7))
fig.patch.set_facecolor('#0f1117')

# ── Left panel: Token sequence diagram ──────────────────────────────────────
ax1 = fig.add_subplot(1, 3, 1)
ax1.set_facecolor('#0f1117')
ax1.set_xlim(-0.5, 11)
ax1.set_ylim(-1, 12)
ax1.axis('off')
ax1.set_title('Token Sequence', color='white', fontsize=12, fontweight='bold')

TOKEN_H = 0.7
TOKEN_W = 0.85
COLORS = {'cls': '#c44dff', 'reg': '#ff8c00', 'patch': '#1a6b3a'}

def draw_token(ax, x, y, label, color, width=0.85):
    b = FancyBboxPatch((x - width/2, y - TOKEN_H/2), width, TOKEN_H,
                        boxstyle='round,pad=0.04', facecolor=color,
                        edgecolor='white', linewidth=0.8, alpha=0.9)
    ax.add_patch(b)
    ax.text(x, y, label, ha='center', va='center', color='white',
            fontsize=8, fontweight='bold')

# DINO sequence: CLS + 4 regs + patches
tokens = [('CLS', COLORS['cls'])] + \
         [(f'R{i}', COLORS['reg']) for i in range(1, 5)] + \
         [(f'P{i}', COLORS['patch']) for i in range(1, 6)] + \
         [('...', '#333')]

for i, (label, color) in enumerate(tokens):
    y = 10 - i * 0.9
    draw_token(ax1, 4, y, label, color)

# Bracket for registers
ax1.annotate('', xy=(5.6, 10 - 0.9), xytext=(5.6, 10 - 0.9*4),
             arrowprops=dict(arrowstyle='<->', color='#ff8c00', lw=1.5))
ax1.text(6.5, 10 - 0.9*2.5, 'Registers\ndiscarded\nat output',
         ha='left', va='center', color='#ff8c00', fontsize=8)

# Legend
legend_items = [mpatches.Patch(color=v, label=k.title()) for k, v in COLORS.items()]
ax1.legend(handles=legend_items, loc='lower left', framealpha=0.3,
           labelcolor='white', fontsize=8)

ax1.text(4, -0.5, '[CLS | Reg×4 | Patch×N]', ha='center', color='#aaa', fontsize=8)

# ── Middle panel: LayerScale diagram ─────────────────────────────────────────
ax2 = fig.add_subplot(1, 3, 2)
ax2.set_facecolor('#0f1117')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('LayerScale in DINOBlock', color='white', fontsize=12, fontweight='bold')

def box(ax, x, y, w, h, txt, sub='', fc='#1a3a5c'):
    p = FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle='round,pad=0.05',
                        facecolor=fc, edgecolor='#4a9eff', linewidth=1.2)
    ax.add_patch(p)
    ax.text(x, y + (0.15 if sub else 0), txt, ha='center', va='center',
            color='white', fontsize=8.5, fontweight='bold')
    if sub:
        ax.text(x, y-0.25, sub, ha='center', va='center', color='#aaa', fontsize=7)

def arr(ax, x, y1, y2):
    ax.annotate('', xy=(x, y2+0.05), xytext=(x, y1-0.05),
                arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.5))

# Input
box(ax2, 5, 9.3, 5, 0.6, 'x  [B, N+5, D]', '', '#1a3a5c')
arr(ax2, 5, 9.0, 8.55)
box(ax2, 5, 8.3, 5, 0.55, 'LayerNorm(x)', '', '#2a2a4a')
arr(ax2, 5, 8.05, 7.6)
box(ax2, 5, 7.35, 5, 0.55, 'DINOAttention (Flash)', 'Q,K,V → softmax(QKᵀ/√d)·V', '#1a3a5c')
arr(ax2, 5, 7.1, 6.65)
# LayerScale box highlighted
box(ax2, 5, 6.4, 5, 0.55, 'LayerScale  γ₁ ⊙ (·)', 'γ ∈ ℝᴰ, init=1e-5', '#3a1a5a')
# Residual
ax2.annotate('', xy=(5, 5.85), xytext=(5, 6.13),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.5))
ax2.annotate('', xy=(2, 6.1), xytext=(2, 9.0),
             arrowprops=dict(arrowstyle='-', color='#ff9944', lw=1.5))
ax2.annotate('', xy=(3.8, 5.85), xytext=(2, 5.85),
             arrowprops=dict(arrowstyle='->', color='#ff9944', lw=1.5))
ax2.text(1.3, 7.5, 'residual\n(skip)', ha='center', va='center', color='#ff9944', fontsize=7.5)
box(ax2, 5, 5.6, 5, 0.55, 'x = x + LayerScale(Attn(LN(x)))', '', '#1a4a3a')

arr(ax2, 5, 5.33, 4.85)
box(ax2, 5, 4.6, 5, 0.55, 'LayerNorm(x)', '', '#2a2a4a')
arr(ax2, 5, 4.33, 3.85)
box(ax2, 5, 3.6, 5, 0.55, 'ViTMLP  (4× GELU)', '', '#1a3a5c')
arr(ax2, 5, 3.33, 2.85)
box(ax2, 5, 2.6, 5, 0.55, 'LayerScale  γ₂ ⊙ (·)', 'independent γ for MLP branch', '#3a1a5a')
ax2.text(0.5, 2.6, '+', ha='center', va='center', color='#ff9944', fontsize=16, fontweight='bold')
arr(ax2, 5, 2.33, 1.85)
box(ax2, 5, 1.6, 5, 0.55, 'x = x + LayerScale(MLP(LN(x)))', '', '#1a4a3a')

# ── Right: Attention comparison DINO vs plain ViT ────────────────────────────
ax3 = fig.add_subplot(1, 3, 3)
ax3.set_facecolor('#0f1117')

rng = np.random.default_rng(7)
# Simulate: ViT without registers — some patch tokens develop high-norm artifacts
grid = 16
N = grid * grid
# Patch token norms — some outliers
norms_vit = np.abs(rng.standard_normal(N)) * 15 + 5
artifact_idx = rng.choice(N, 12, replace=False)
norms_vit[artifact_idx] += rng.uniform(50, 100, 12)  # artifact tokens
norms_vit_img = norms_vit.reshape(grid, grid)

# DINO: artifacts absorbed by register tokens, patch norms more uniform
norms_dino = np.abs(rng.standard_normal(N)) * 12 + 5
norms_dino_img = norms_dino.reshape(grid, grid)

fig2, (axA, axB) = plt.subplots(1, 2, figsize=(9, 4))
fig2.patch.set_facecolor('#0f1117')
fig2.suptitle('Patch Token Norms: ViT vs DINOv2', color='white', fontsize=12)
for axi, data, title in zip([axA, axB],
                              [norms_vit_img, norms_dino_img],
                              ['Plain ViT — artifact tokens (bright)', 'DINOv2 — artifacts in registers']):
    axi.set_facecolor('#0f1117')
    im = axi.imshow(data, cmap='hot', interpolation='nearest')
    plt.colorbar(im, ax=axi, fraction=0.046)
    axi.set_title(title, color='white', fontsize=9)
    axi.tick_params(colors='white')
    for sp in axi.spines.values():
        sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig('dino_register_tokens.png', dpi=120, bbox_inches='tight',
            facecolor=fig2.get_facecolor())
plt.show()

plt.figure(fig.number)
plt.tight_layout()
plt.savefig('dino_architecture.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Diagrams saved.')

## Key Innovations

### 1. Register Tokens — Eliminating Artifact Tokens

**Problem:** Large ViTs trained with DINO develop high-norm "artifact" tokens in patch positions that correspond to smooth background regions (sky, grass, uniform textures). These tokens accumulate global image statistics and disrupt local feature quality.

**Solution:** Introduce $R$ learnable register tokens (default $R=4$), inserted between CLS and the patch tokens **after** positional embedding:
$$\text{sequence} = [x_{cls}, \; r_1, r_2, r_3, r_4, \; x_1, \ldots, x_N]$$

Register tokens are **not** given positional embeddings (they represent global context, not spatial positions). At inference, they are simply **discarded** from the output.

**Result:** Artifact behavior moves to the register tokens, which are designed to hold global info. Patch tokens retain clean local spatial features.

### 2. LayerScale — Stabilising Deep Transformers

Introduced in CaiT (Touvron et al., 2021) and adopted by DINOv2:

$$x \leftarrow x + \gamma_1 \odot \text{Attention}(\text{LN}(x))$$
$$x \leftarrow x + \gamma_2 \odot \text{MLP}(\text{LN}(x))$$

where $\gamma_1, \gamma_2 \in \mathbb{R}^D$ are **per-channel learnable parameters** initialised at $10^{-5}$.

**Why it works:** Near $\gamma \approx 0$, residuals start near-zero, preventing gradient explosion in the first steps of training. As training progresses, $\gamma$ grows, allowing residuals to contribute more. This is a form of adaptive skip-connection strength.

### 3. Flash Attention — Memory-Efficient Scaled Dot-Product

PyTorch 2.0 provides `F.scaled_dot_product_attention`, which:
- Fuses softmax + matmul into a single CUDA kernel (Flash Attention algorithm)
- Avoids materialising the full $N \times N$ attention matrix in HBM
- Gives **~2× speedup** and **~10× memory reduction** for long sequences
- Result is numerically equivalent to standard attention

```python
# Standard attention (materialises [B,H,N,N] attention matrix)
attn = softmax(q @ k.T / sqrt(d))
out  = attn @ v  # O(N²) memory

# Flash Attention (fused, O(N) memory)
out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
```

## Module Walkthrough

### LayerScale
```python
class LayerScale(nn.Module):
    def __init__(self, dim, init_value=1e-5):
        # γ: per-channel scalar, starts near zero
        self.gamma = nn.Parameter(init_value * torch.ones(dim))

    def forward(self, x):   # x: [B, N, D]
        return x * self.gamma  # element-wise broadcast over D
```

### DINOBlock — Full Residual with LayerScale
```python
class DINOBlock(nn.Module):
    def forward(self, x, return_attn=False):
        attn_out, attn_weights = self.attn(self.norm1(x), return_attn=return_attn)
        x = x + self.ls1(attn_out)          # scaled residual
        x = x + self.ls2(self.mlp(self.norm2(x)))
        return x, attn_weights
```

### Forward — Register token insertion and removal
```python
def forward(self, x):
    patches = self.patch_embed(x)                # [B, N, D]
    cls = self.cls_token.expand(B, -1, -1)       # [B, 1, D]
    tokens = torch.cat([cls, patches], dim=1)    # [B, N+1, D]
    tokens = tokens + self.pos_embed             # positional info for CLS+patches

    # Insert 4 register tokens (no positional embedding for them)
    regs   = self.register_tokens.expand(B, -1, -1)  # [B, 4, D]
    tokens = torch.cat([tokens[:, :1],   # CLS
                         regs,            # 4 registers
                         tokens[:, 1:]], dim=1)  # patches

    for blk in self.blocks:
        tokens, attn = blk(tokens)
    tokens = self.norm(tokens)

    # Discard register tokens (positions 1..4)
    pooled   = tokens[:, 0]           # CLS
    patches  = tokens[:, 1+n_reg:]    # N patch tokens
    return BackboneOutput(pooled, patches, [spatial_reshape(patches)])
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

backbone = build_vision_backbone('dino_vit_small', img_size=224, return_all_attn=True)
backbone.eval()
print(f'Parameters:       {backbone.num_parameters():,}')
print(f'Embed dim:        {backbone.embed_dim}')
print(f'Register tokens:  {backbone.n_register}')
print(f'Patch size:       {backbone.patch_size}')  # 14 for DINOv2

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print(f'\npooled_output:  {out.pooled_output.shape}')   # [1, 384]
print(f'patch_tokens:   {out.patch_tokens.shape}')      # [1, 256, 384] (16×16 grid at p=14)
print(f'feature_maps:   {out.feature_maps[0].shape}')   # [1, 384, 16, 16]
print(f'\n[Register tokens were inserted during forward but discarded from output]')

# Visualise DINOv2-style attention maps with high-quality patches
fig, axes = plt.subplots(1, 6, figsize=(15, 3.5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('DINOv2-S: CLS Attention Maps per Head (Last Block)', color='white', fontsize=12)

# Last block attention, CLS row, patch columns
last_attn  = out.attentions[-1][0]   # [H, N+1+n_reg, N+1+n_reg]
n_reg = backbone.n_register
n_total = last_attn.shape[-1]
# CLS attends to patches (skip register columns)
# token order: CLS=0, reg=1..4, patches=5..
cls_to_patches = last_attn[:, 0, 1+n_reg:].reshape(backbone.num_heads, 16, 16)
cls_to_patches = cls_to_patches.numpy()

for i, ax in enumerate(axes):
    ax.set_facecolor('#0f1117')
    im = ax.imshow(cls_to_patches[i], cmap='inferno', interpolation='bilinear', vmin=0)
    ax.set_title(f'Head {i}', color='white', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('dino_attn_maps.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Note: each head focuses on different semantic regions (object parts, background, edges).')

## References

### Primary
1. M. Oquab et al., "DINOv2: Learning Robust Visual Features without Supervision," *TMLR 2024*. [arXiv:2304.07193](https://arxiv.org/abs/2304.07193)
2. T. Darcet et al., "Vision Transformers Need Registers," *ICLR 2024*. [arXiv:2309.16588](https://arxiv.org/abs/2309.16588) — register tokens paper

### Related Works
3. M. Caron et al., "Emerging Properties in Self-Supervised Vision Transformers," *ICCV 2021*. [arXiv:2104.14294](https://arxiv.org/abs/2104.14294) — DINO v1
4. H. Touvron et al., "Going Deeper with Image Transformers," *ICCV 2021*. [arXiv:2103.17239](https://arxiv.org/abs/2103.17239) — LayerScale (CaiT)
5. T. Dao et al., "FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning," *ICLR 2024*. [arXiv:2307.08691](https://arxiv.org/abs/2307.08691)
6. A. Dosovitskiy et al., "An Image is Worth 16×16 Words," *ICLR 2021*. [arXiv:2010.11929](https://arxiv.org/abs/2010.11929) — ViT
7. J. Assran et al., "Self-Supervised Learning from Images with a Joint-Embedding Predictive Architecture," *CVPR 2023*. [arXiv:2301.08243](https://arxiv.org/abs/2301.08243) — I-JEPA